# 02 · Build clean tables & explore

We run the SQL transforms (portable across DuckDB and Snowflake) to create
`stg_games`, `team_game_log`, `team_season`, and `player_season_scoring`, then
sanity-check the data: top scorers and the spread of team scoring.

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
os.chdir(ROOT)
import pandas as pd
pd.set_option('display.float_format', lambda x: f'{x:.3f}')

In [ ]:
from liiga.transform import run_transforms
run_transforms()

In [ ]:
from liiga.db import get_connection, query_df
con = get_connection()
print('Top goal scorers, 2025-26:')
display(query_df(con, '''
  SELECT first_name, last_name, team, goals, assists
  FROM player_season_scoring WHERE season=2026
  ORDER BY goals DESC LIMIT 10'''))

In [ ]:
import matplotlib.pyplot as plt
gf = query_df(con, '''SELECT team, gf_per_game FROM team_season
                      WHERE season=2026 ORDER BY gf_per_game DESC''')
con.close()
ax = gf.plot.bar(x='team', y='gf_per_game', legend=False, figsize=(10,4),
                 title='Goals for per game by team (2025-26)')
ax.set_ylabel('goals / game'); plt.tight_layout(); plt.show()

Team scoring ranges roughly 1.8–3.8 goals/game — keep this spread in mind; our projected team strengths should land in a similar range.